In [ ]:
from msal import ConfidentialClientApplication
import requests
import pandas as pd
import io

In [ ]:
# --- Configuration ---
CLIENT_ID = "your-azure-app-client-id"
CLIENT_SECRET = "your-azure-app-client-secret"
TENANT_ID = "5fe78ac1-1afe-4009-aa04-a71efb4a5042"
USER_EMAIL = "aduragbemioyinlola@outlook.com"

AUTHORITY = f"https://login.microsoftonline.com/{TENANT_ID}"
SCOPE = ["https://graph.microsoft.com/.default"]
GRAPH_ENDPOINT = "https://graph.microsoft.com/v1.0"

In [ ]:
def get_access_token() -> str:
    """Acquire an OAuth2 access token via client credentials."""
    app = ConfidentialClientApplication(
        CLIENT_ID,
        authority=AUTHORITY,
        client_credential=CLIENT_SECRET,
    )
    result = app.acquire_token_silent(SCOPE, account=None)
    if not result:
        result = app.acquire_token_for_client(scopes=SCOPE)
    if "access_token" not in result:
        raise Exception(f"Could not acquire token: {result.get('error_description')}")
    return result["access_token"]


def get_csv_attachment(message_id: str) -> pd.DataFrame | None:
    """
    Fetch the first CSV attachment from a specific email.

    Args:
        message_id: The Graph API message ID.

    Returns:
        A pandas DataFrame of the CSV contents, or None if no CSV found.
    """
    token = get_access_token()
    headers = {"Authorization": f"Bearer {token}"}

    # Step 1: List all attachments on the email
    url = f"{GRAPH_ENDPOINT}/users/{USER_EMAIL}/messages/{message_id}/attachments"
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    attachments = response.json().get("value", [])

    for attachment in attachments:
        name = attachment.get("name", "")
        content_type = attachment.get("contentType", "")

        # Step 2: Filter for CSV files by name extension or MIME type
        is_csv = name.lower().endswith(".csv") or "csv" in content_type.lower()

        if is_csv:
            attachment_id = attachment["id"]

            # Step 3: Fetch the raw attachment bytes
            att_url = (
                f"{GRAPH_ENDPOINT}/users/{USER_EMAIL}"
                f"/messages/{message_id}/attachments/{attachment_id}"
                f"/$value"  # /$value returns the raw binary content
            )
            att_response = requests.get(att_url, headers=headers)
            att_response.raise_for_status()

            # Step 4: Decode bytes and load into a DataFrame
            csv_bytes = att_response.content
            df = pd.read_csv(io.BytesIO(csv_bytes))

            print(f"Attachment found: '{name}' ({len(df)} rows, {len(df.columns)} columns)")
            return df

    print("No CSV attachment found on this email.")
    return None


def fetch_email_and_csv(subject: str) -> pd.DataFrame | None:
    """
    Find the most recent email matching the subject and return
    its CSV attachment as a DataFrame.

    Args:
        subject: Email subject to search for (e.g. "sleeping cells").

    Returns:
        DataFrame of the CSV contents, or None if not found.
    """
    token = get_access_token()
    headers = {"Authorization": f"Bearer {token}"}

    # Search for the email by subject
    url = (
        f"{GRAPH_ENDPOINT}/users/{USER_EMAIL}/messages"
        f"?$search=\"subject:{subject}\""
        f"&$top=5"
        f"&$select=id,subject,receivedDateTime,hasAttachments"
        f"&$orderby=receivedDateTime desc"
    )
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    messages = response.json().get("value", [])

    # Find the first email that actually has attachments
    for msg in messages:
        if not msg.get("hasAttachments"):
            continue

        print(f"Found email: '{msg['subject']}' received at {msg['receivedDateTime']}")
        df = get_csv_attachment(msg["id"])
        if df is not None:
            return df

    print(f"No email with a CSV attachment found for subject: '{subject}'")
    return None


# --- Example usage ---
if __name__ == "__main__":
    df = fetch_email_and_csv("sleeping cells")

    if df is not None:
        print("\nFirst 5 rows:")
        print(df.head())

        # From here you can feed directly into your CV pipeline
        # e.g. image_paths = df["image_path"].tolist()